# Domain01: カメラキャリブレーション練習ノートブック

このノートブックはステレオ／単眼キャリブレーションの基礎演習用です。
目的:
- ChArUco/チェスボード画像から内部パラメータを推定する
- 再投影誤差を可視化して品質評価する
- 校正結果を YAML で保存し、後続パイプラインで使える形にする

※ 画像パスやボードパラメータは実環境に合わせて入力してください。

In [5]:
# Section 1: ライブラリのインポート

# 必要に応じて環境でインストールしてください（例: pip install opencv-contrib-python pyyaml）
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml

from tools.logger import log

print('imports ok')

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Section 2: データ読み込み

# ここに画像フォルダを指定してください
image_dir = '../sandbox/Step01 Markerless mocap/sample_calib_images/'  # <-- 実際のパスに変更
pattern = os.path.join(image_dir, '*.jpg')
image_files = sorted(glob.glob(pattern))

print(f'found {len(image_files)} images')
# サンプル表示
if len(image_files)>0:
    img = cv2.imread(image_files[0])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,4)); plt.imshow(img_rgb); plt.axis('off')
else:
    print('No images found — set `image_dir` to your images')

In [ ]:
# Section 3: データ前処理 / ChArUco 検出関数

ARUCO_DICT = cv2.aruco.Dictionary_get(cv2.aruco.DICT_5X5_1000)

def create_charuco_board(squares_x=7, squares_y=5, square_length=0.025, marker_length=0.019):
    """Create and return a ChArUco board and charuco detector parameters.
    square_length and marker_length in meters (or consistent unit)
    """
    board = cv2.aruco.CharucoBoard_create(squares_x, squares_y, square_length, marker_length, ARUCO_DICT)
    return board


def detect_charuco_corners(image, board, detector_params=None):
    """Detect ChArUco corners in a single image.
    Returns retval, charuco_corners, charuco_ids
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    corners, ids, rejected = cv2.aruco.detectMarkers(gray, ARUCO_DICT)
    if len(corners)>0:
        cv2.aruco.refineDetectedMarkers(gray, board, corners, ids, rejected)
        retval, charuco_corners, charuco_ids = cv2.aruco.interpolateCornersCharuco(corners, ids, gray, board)
        return retval, charuco_corners, charuco_ids
    return 0, None, None


# バッチ処理の例
board = create_charuco_board(squares_x=7, squares_y=5, square_length=0.069, marker_length=0.045)
objpoints = []
imgpoints = []
for p in image_files:
    img = cv2.imread(p)
    r, cc, ids = detect_charuco_corners(img, board)
    if r>0:
        imgpoints.append(cc)
        objpoints.append(board.chessboardCorners[:len(cc)])

print('detected in', len(imgpoints), 'images')

In [ ]:
# Section 4: 特徴量エンジニアリング（キャリブ用のobject points）

def make_objpoints_for_charuco(board, detected_corners):
    """Given a Charuco board and detected charuco corners, return corresponding 3D object points.
    This uses board.chessboardCorners as the canonical corner locations.
    """
    # board.chessboardCorners contains the 3D locations (Nx3) in board coordinate units
    # detected_corners.shape = (N,1,2) -> we assume ordering matches subset of board corners
    return board.chessboardCorners[:len(detected_corners)]

# サンプル: objpoints, imgpoints は上で作成済み


In [ ]:
# Section 5: 訓練/検証データの分割（キャリブ画像の分割例）

# キャリブ画像の一部を検証用に取っておく例
from sklearn.model_selection import train_test_split

indices = list(range(len(imgpoints)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
print('train', len(train_idx), 'val', len(val_idx))

# 実行例（あとでcalibrateを呼ぶ時にtrain側のobjpoints,imgpointsを使う）

In [ ]:
# Section 6: モデル定義（キャリブでは最適化関数/パラメータの定義）

def calibrate_single_camera(objpoints, imgpoints, image_size, flags=cv2.CALIB_RATIONAL_MODEL):
    """Wrapper for cv2.calibrateCamera for a single camera.
    Returns ret, mtx, dist, rvecs, tvecs
    """
    if len(objpoints)==0 or len(imgpoints)==0:
        raise ValueError('objpoints/imgpoints empty')
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, image_size, None, None, flags=flags)
    return ret, mtx, dist, rvecs, tvecs


def save_camera_yaml(path, mtx, dist):
    data = {'camera_matrix': mtx.tolist(), 'dist_coeff': dist.tolist()}
    with open(path, 'w') as f:
        yaml.safe_dump(data, f)
    print('saved', path)


In [ ]:
# Section 7: モデル訓練（キャリブ実行）

# 実行例: 実行前に image_size を正しく設定
if len(train_idx)>0:
    sample_img = cv2.imread(image_files[train_idx[0]])
    h, w = sample_img.shape[:2]
    image_size = (w,h)
    # calibrate
    ret, mtx, dist, rvecs, tvecs = calibrate_single_camera([objpoints[i] for i in train_idx], [imgpoints[i] for i in train_idx], image_size)
    print('RMS:', ret)
    log(f'calibrate_single_camera RMS={ret}')
    save_camera_yaml('../calibration_results/camera1.yaml', mtx, dist)
else:
    print('No train images available for calibration')

In [ ]:
# Section 8: モデル評価（再投影誤差の可視化）

def compute_reprojection_error(objpoints, imgpoints, rvecs, tvecs, mtx, dist):
    tot_err = 0
    errs = []
    for i in range(len(objpoints)):
        imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
        err = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2)/len(imgpoints2)
        errs.append(err)
        tot_err += err
    return tot_err/len(objpoints), errs

# 可視化例
try:
    mean_err, errs = compute_reprojection_error([objpoints[i] for i in train_idx], [imgpoints[i] for i in train_idx], rvecs, tvecs, mtx, dist)
    print('mean reproj error', mean_err)
    plt.figure(); plt.plot(errs); plt.title('per-image reprojection error'); plt.xlabel('image'); plt.ylabel('px')
except Exception as e:
    print('Run calibration first to compute reprojection error', e)


In [ ]:
# Section 9: モデル保存と読み込み

# 保存は先ほどの save_camera_yaml を使用

# 読み込み例
with open('../calibration_results/camera1.yaml', 'r') as f:
    cam = yaml.safe_load(f)
    print('loaded camera', 'matrix keys:', cam.keys())


In [ ]:
# Section 10: 可視化（歪み補正のプレビュー）

try:
    sample = cv2.imread(image_files[0])
    und = cv2.undistort(sample, mtx, dist)
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)); plt.title('orig'); plt.axis('off')
    plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(und, cv2.COLOR_BGR2RGB)); plt.title('undistorted'); plt.axis('off')
except Exception as e:
    print('Run calibration first to visualize undistort:', e)


In [ ]:
# Section 11: 単体テスト（簡易チェック）

def _test_read_images():
    assert isinstance(image_files, list)

def _test_detect_function_runs():
    if len(image_files)>0:
        img = cv2.imread(image_files[0])
        r, cc, ids = detect_charuco_corners(img, board)
        assert r>=0

print('unit tests defined: run with pytest or call functions directly')
